# CTR prediction (Wide & Deep, DeepFM, DLRM) — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def cosine(a,b):
    a=np.asarray(a,dtype=float); b=np.asarray(b,dtype=float)
    return float(a@b/(np.linalg.norm(a)*np.linalg.norm(b)))
def sigmoid(x): return 1/(1+np.exp(-np.asarray(x)))
def dcg(rels): return sum((2**r-1)/math.log2(i+2) for i,r in enumerate(rels))
def ndcg(rels):
    best=sorted(rels, reverse=True)
    return dcg(rels)/dcg(best) if dcg(best)>0 else 0.0
print('setup ok')

## Predicting click probability from sparse fields and interactions
CTR models turn categorical fields into embeddings, cross them, and feed a calibrated probability. The examples compute logistic CTR, a wide cross, an FM interaction, a tiny DeepFM sum, and calibration bins.

In [ ]:
# 1) logistic CTR turns a logit into a probability
z= -1.2 + 0.7 + 0.4; p=float(sigmoid(z))
plt.figure(figsize=(6,3)); plt.bar(['logit','CTR'],[z,p]); plt.title(f'z={z:.3f}, p={p:.3f}')
assert abs(p-0.47502081252106)<1e-12

In [ ]:
# 2) a wide crossed feature memorizes a useful advertiser-item pair
base=-2.0; ad=0.5; item=0.4; cross=1.2
plain=float(sigmoid(base+ad+item)); wide=float(sigmoid(base+ad+item+cross))
plt.figure(figsize=(6,3)); plt.bar(['without cross','with cross'],[plain,wide]); plt.title('wide feature boosts known pair')
assert wide>plain and abs(wide-0.52497918747894)<1e-12

In [ ]:
# 3) factorization-machine interaction is sum of pairwise embedding dots
E=np.array([[0.2,0.5],[0.4,0.1],[0.3,0.6]])
pairs=[float(E[i]@E[j]) for i in range(3) for j in range(i+1,3)]
fm=sum(pairs)
plt.figure(figsize=(6,3)); plt.bar(['0-1','0-2','1-2'],pairs); plt.title(f'FM interaction = {fm:.3f}')
assert abs(fm-0.67)<1e-12

In [ ]:
# 4) DeepFM adds linear, FM, and deep terms before sigmoid
linear=-0.3; fm=0.67; deep=0.25; p=float(sigmoid(linear+fm+deep))
plt.figure(figsize=(6,3)); plt.bar(['linear','FM','deep'],[linear,fm,deep]); plt.title(f'CTR = sigmoid(0.620) = {p:.3f}')
assert abs(p-0.6502185485738271)<1e-12

In [ ]:
# 5) calibration: predicted probabilities should match observed click rates by bin
pred=np.array([.05,.08,.12,.18,.22,.28,.35,.42]); click=np.array([0,0,0,0,1,0,1,1])
bins=np.array([0,.2,.4,.6]); obs=[]; exp=[]
for lo,hi in zip(bins[:-1],bins[1:]):
    m=(pred>=lo)&(pred<hi); exp.append(pred[m].mean()); obs.append(click[m].mean())
plt.figure(figsize=(5,4)); plt.plot(exp,obs,'o-'); plt.plot([0,.5],[0,.5],'k--'); plt.xlabel('mean predicted'); plt.ylabel('observed CTR'); plt.title('calibration bins')
assert len(obs)==3 and obs[0]==0.0 and obs[-1]==1.0